In [ ]:
import os
import numpy as np
import pandas as pd
import datetime
import pickle
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, PowerTransformer

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# 1. Folder structure setup
horizons = [6, 12, 24]
feature_types = ['UNIVARIATE', 'MULTIVARIATE']

for ft in feature_types:
    for h in horizons:
        base_path = f"{ft}/{h}h"
        os.makedirs(f"{base_path}/dataset_ready", exist_ok=True)
        os.makedirs(f"{base_path}/LOSS", exist_ok=True)
        os.makedirs(f"{base_path}/MODELS", exist_ok=True)
        os.makedirs(f"{base_path}/results", exist_ok=True)
print("Folder structure created successfully!")

In [ ]:
# 2. Data Loading & Setup
df = pd.read_csv("df_combined_AT.csv")
df_dummy = df.drop(columns=['utc_timestamp'])

# Menentukan batas train (80%)
train_idx = int(df_dummy.shape[0] * 0.8)

# Normalisasi menggunakan Yeo-Johnson untuk fitur berdistribusi skew
skewed_cols = ['rain (mm)', 'relative_humidity_2m (%)', 'sunshine_duration (s)']
pt = PowerTransformer(method='yeo-johnson')

# Fit Yeo-Johnson HANYA pada data train untuk mencegah data leakage
pt.fit(df_dummy[skewed_cols].iloc[:train_idx])

# Aplikasikan transformasi ke seluruh dataset
df_dummy[skewed_cols] = pt.transform(df_dummy[skewed_cols])

# Setelah Yeo-Johnson, aplikasikan MinMax Scaling ke seluruh fitur
scaler = MinMaxScaler()
scaler.fit(df_dummy.iloc[:train_idx]) 
scaled_data = scaler.transform(df_dummy)

# Simpan transformers dan data
with open('scaler_yeojohnson.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('power_transformer_yeojohnson.pkl', 'wb') as f:
    pickle.dump(pt, f)
np.save('scaled_data_yeojohnson.npy', scaled_data)

print("Data loaded, Yeo-Johnson Transformed, and MinMax Scaled! Shape:", scaled_data.shape)

In [ ]:
# Visualisasi distribusi setelah Yeo-Johnson dan Scaling
plt.figure(figsize=(15, 10))
for i, col in enumerate(df_dummy.columns):
    plt.subplot(len(df_dummy.columns) // 3 + 1, 3, i + 1)
    sns.histplot(scaled_data[:, i], bins=50, kde=True)
    plt.title(f'Distribusi Scaled: {col}')
    plt.xlabel('Nilai')
    plt.ylabel('Frekuensi')

plt.tight_layout()
plt.show()

display(pd.DataFrame(scaled_data, columns=df_dummy.columns).describe())

In [ ]:
# 3. Helper Functions
def create_supervised_data(scaled_data_x, scaled_data_y, lag, horizon):
    X, y = [], []
    for i in range(lag, len(scaled_data_x) - horizon + 1):
        X.append(scaled_data_x[i - lag:i, :]) 
        y.append(scaled_data_y[i:i + horizon]) 
    return np.array(X), np.array(y)

def inverse_transform_target(y_scaled, scaler, target_col_idx=0, total_cols=5):
    '''
    Descaling: Denormalisasi kembali ke skala aktual (MW).
    Target Load hanya diskalakan menggunakan MinMaxScaler (tidak menggunakan Yeo-Johnson),
    sehingga kita hanya perlu menggunakan inverse_transform dari scaler.
    '''
    if len(y_scaled) == 0:
        return y_scaled
    original_shape = y_scaled.shape
    y_flat = y_scaled.flatten()
    
    # Buat matrix dummy karena scaler memerlukan 5 fitur
    dummy_matrix = np.zeros((len(y_flat), total_cols))
    dummy_matrix[:, target_col_idx] = y_flat
    
    y_inv = scaler.inverse_transform(dummy_matrix)[:, target_col_idx]
    return y_inv.reshape(original_shape)

def get_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    if not np.any(non_zero): return np.nan
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

def get_row_mapes(y_true, y_pred):
    mapes = []
    for i in range(len(y_true)):
        non_zero = y_true[i] != 0
        if np.any(non_zero):
            m = np.mean(np.abs((y_true[i][non_zero] - y_pred[i][non_zero]) / y_true[i][non_zero])) * 100
        else:
            m = np.nan
        mapes.append(m)
    return mapes

In [ ]:
# 4. Model Architectures
def build_lstm(seq_len, num_d, out_steps):
    inputs = layers.Input(shape=(seq_len, num_d))
    x = layers.LSTM(64, return_sequences=False)(inputs)
    outputs = layers.Dense(out_steps)(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="LSTM")
    model.compile(optimizer='adam', loss='mse')
    return model

def build_bilstm(seq_len, num_d, out_steps):
    inputs = layers.Input(shape=(seq_len, num_d))
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(inputs)
    outputs = layers.Dense(out_steps)(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="BiLSTM")
    model.compile(optimizer='adam', loss='mse')
    return model

def build_attention(seq_len, num_d, out_steps):
    inputs = layers.Input(shape=(seq_len, num_d))
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    attention_output = layers.MultiHeadAttention(key_dim=64, num_heads=4, dropout=0.2)(x, x)
    x = layers.Add()([attention_output, inputs])
    x2 = layers.LayerNormalization(epsilon=1e-6)(x)
    x2 = layers.Dense(64, activation="relu")(x2)
    x2 = layers.Dropout(0.2)(x2)
    x2 = layers.Dense(num_d)(x2)
    x = layers.Add()([x2, x])
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(out_steps)(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="Attention")
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# 5. Massive Experiment Loop (With Descaling)
epochs = 200
batch_size = 32
lag = 24  # Standard lag looking back 24 hours

total_cols = scaled_data.shape[1]

for ft in feature_types:
    for h in horizons:
        print(f"\n{'='*60}\n--> STARTING EXPERIMENT: Feature = {ft} | Horizon = {h}h\n{'='*60}")
        base_path = f"{ft}/{h}h"
        
        # Determine X and y shapes based on Univariate vs Multivariate
        if ft == 'UNIVARIATE':
            scaled_data_x = scaled_data[:, 0:1] 
        else:
            scaled_data_x = scaled_data         
        
        scaled_data_y = scaled_data[:, 0]       
        
        # Sliding windows
        X_all, y_all = create_supervised_data(scaled_data_x, scaled_data_y, lag=lag, horizon=h)
        
        test_split_idx = train_idx - lag
        
        X_train_val = X_all[:test_split_idx]
        y_train_val = y_all[:test_split_idx]
        
        X_test = X_all[test_split_idx:]
        y_test = y_all[test_split_idx:]
        
        val_split_idx = int(len(X_train_val) * 0.8)
        X_train, y_train = X_train_val[:val_split_idx], y_train_val[:val_split_idx]
        X_val, y_val = X_train_val[val_split_idx:], y_train_val[val_split_idx:]
        
        # Save datasets (custom name to prevent overwrite)
        np.save(f"{base_path}/dataset_ready/X_train_yj.npy", X_train)
        np.save(f"{base_path}/dataset_ready/y_train_yj.npy", y_train)
        np.save(f"{base_path}/dataset_ready/X_val_yj.npy", X_val)
        np.save(f"{base_path}/dataset_ready/y_val_yj.npy", y_val)
        np.save(f"{base_path}/dataset_ready/X_test_yj.npy", X_test)
        np.save(f"{base_path}/dataset_ready/y_test_yj.npy", y_test)
        
        seq_len = X_train.shape[1]
        num_d = X_train.shape[2]
        out_steps = y_train.shape[1]
        
        test_start_idx_for_ts = train_idx
        test_timestamps = df['utc_timestamp'].iloc[test_start_idx_for_ts : test_start_idx_for_ts + len(y_test)].values
        
        models_dict = {
            "LSTM": build_lstm(seq_len, num_d, out_steps),
            "BiLSTM": build_bilstm(seq_len, num_d, out_steps),
            "Attention": build_attention(seq_len, num_d, out_steps)
        }
        
        for name, model in models_dict.items():
            print(f"\n--- Training {name} ({ft} - {h}h) ---")
            
            history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, verbose=2)
            
            train_loss = history.history['loss']
            val_loss = history.history['val_loss']
            np.save(f"{base_path}/LOSS/loss_{name}_yj.npy", np.array([train_loss, val_loss]))
            
            plt.figure(figsize=(8, 5))
            plt.plot(train_loss, label='Train Loss')
            plt.plot(val_loss, label='Val Loss')
            plt.title(f'{name} ({ft} - {h}h) Loss')
            plt.legend()
            plt.grid()
            plt.savefig(f"{base_path}/LOSS/loss_{name}_yj.png")
            plt.close()
            
            model.save(f"{base_path}/MODELS/{name}_yj.h5")
            
            pred_train = model.predict(X_train)
            pred_test = model.predict(X_test)
            
            # ---------------------------------------------
            # DENORMALIZATION (DESCALING) KE SKALA ASLI
            # ---------------------------------------------
            y_train_inv = inverse_transform_target(y_train, scaler, target_col_idx=0, total_cols=total_cols)
            pred_train_inv = inverse_transform_target(pred_train, scaler, target_col_idx=0, total_cols=total_cols)
            
            y_test_inv = inverse_transform_target(y_test, scaler, target_col_idx=0, total_cols=total_cols)
            pred_test_inv = inverse_transform_target(pred_test, scaler, target_col_idx=0, total_cols=total_cols)
            
            # Evaluasi MAPE menggunakan nilai asli Megawatt (MW), bukan 0-1
            mape_train = get_mape(y_train_inv, pred_train_inv)
            mape_test = get_mape(y_test_inv, pred_test_inv)
            
            # Global CSV logging
            date_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            global_csv_path = "global_summary_yeojohnson_denormalized.csv"
            new_row = pd.DataFrame([{
                'datetime_run': date_str,
                'feature_type': ft,
                'horizon': f"{h}h",
                'model_name': name,
                'mape_train_denorm': round(mape_train, 4),
                'mape_test_denorm': round(mape_test, 4)
            }])
            new_row.to_csv(global_csv_path, mode='a', header=not os.path.exists(global_csv_path), index=False)

            print(f"--> {name} Train MAPE (Denorm): {mape_train:.4f} | Test MAPE (Denorm): {mape_test:.4f}")
            
            # CSV Save untuk Test Set
            row_mapes = get_row_mapes(y_test_inv, pred_test_inv)
            
            lag_strs = [str(list(x.flatten())) for x in X_test]
            act_strs = [str(list(y)) for y in y_test_inv]
            pred_strs = [str(list(p)) for p in pred_test_inv]
            
            df_res = pd.DataFrame({
                'DATE_OF_DATA_POINT': test_timestamps,
                'LAG_DATA_SCALED': lag_strs,
                'ACTUAL_VALUE_HORIZON_DENORM': act_strs,
                'PREDICTED_VALUE_HORIZON_DENORM': pred_strs,
                'MAPE_OF_THIS_DATA_POINT': row_mapes
            })
            df_res.to_csv(f"{base_path}/results/{name}_yj_denormalized.csv", index=False)